# Train Segment Classifier — Unified AL Protocol

Four-condition experiment comparing **how human labels are selected and integrated**, holding the labelling protocol (one pair shown at a time, "which is worse?") constant across stages.

| Stage | Init weights | Selection | Iteration |
|---|---|---|---|
| **1 — auto baseline** | scratch | n/a (auto labels only) | one shot |
| **2 — random + warm fine-tune** | from S1 ckpt | uniform random | one shot |
| **3 — AL + warm fine-tune** | from S1 ckpt | `H(σ(s_A − s_B)) · ‖φ_A − φ_B‖` w/ S1 model | one shot |
| **4 — iterative AL** | from S1 ckpt → continue from prev round | same acquisition w/ **current** model | R=4 rounds × 14 labels |

All elicitation pulls from the **train-map** pool (`train_seeds_manifest.txt`, 1340 rollouts on `TRAIN_MAP_SEEDS = {128–143}`). All four stages spend the same 56-label budget. Stages 2/3/4 fine-tune from the Stage 1 baseline (`baseline.pt`).

**Held-out human val signal.** Before any stage runs, you elicit a separate **val-map** label set via random sampling against `val_seeds_manifest.txt` (`VAL_MAP_SEEDS = {144–147}`, 329 rollouts). These labels are passed via `--val-annotations` to every stage's training run. They are *eval-only* — never used as training overrides — and provide the human-aligned generalization metric across all four stages on the same val set.

`annotations_legacy_free_curation.json` is the prior 56-label hand-curated set (humans saw whole rollouts and picked the most-extreme `(worst, clean)` pair). Different protocol, all on train maps — kept for reference, used only in the post-hoc Stage 1 diagnostic.

**Comparison axes:**
- **S2 vs S3:** does AL acquisition beat random selection (one-shot)?
- **S3 vs S4:** does iterative refinement beat one-shot AL?

**Prereq.** Push the branch first if you haven't already:
```
git push -u origin feature/preference_elicitation
```

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
REPO_URL  = "https://github.com/isa-deluis42/MAPF-GPT-DDG.git"
BRANCH    = "feature/preference_elicitation"
REPO_DIR  = "/content/MAPF-GPT-DDG"

DRIVE_ROOT = "/content/drive/MyDrive/mapf_congestion"
OUT_DIR    = f"{DRIVE_ROOT}/out/segment_classifier"
ITERATIVE_DIR = f"{OUT_DIR}/iterative"

DATA_DIR        = f"{REPO_DIR}/dataset/held_out"
TRAIN_MANIFEST  = f"{REPO_DIR}/train_seeds_manifest.txt"
VAL_MANIFEST    = f"{REPO_DIR}/val_seeds_manifest.txt"

# Each stage points --annotations at the JSON for *that* acquisition strategy.
# Val-map annotations are shared across all four stages (the held-out human val signal).
RANDOM_ANNOTATIONS    = f"{REPO_DIR}/annotation_random.json"
WARMSTART_ANNOTATIONS = f"{REPO_DIR}/annotation_elicitation_warmstart.json"
ITERATIVE_ANNOTATIONS = f"{REPO_DIR}/annotation_iterative.json"
VAL_MAP_ANNOTATIONS   = f"{REPO_DIR}/annotation_val_map.json"

BASELINE_CKPT  = f"{OUT_DIR}/baseline.pt"
RANDOM_CKPT    = f"{OUT_DIR}/random_baseline.pt"
WARMSTART_CKPT = f"{OUT_DIR}/warmstart_elicitation.pt"
# Stage 4 iterative produces R checkpoints: ITERATIVE_DIR/round_{r}.pt for r=1..N_ROUNDS

LEGACY_ANNOTATIONS = f"{REPO_DIR}/annotations_legacy_free_curation.json"

EPOCHS          = 32   # Stages 1/2/3
BATCH_SIZE      = 128
LR              = 3e-4
CONTEXT_SEGMENTS = 1   # 1 = this segment only; 2 = this + next as future context
BASE_CH         = 8
MIN_CHECKPOINT  = 0    # 0 = use all rollouts; >0 keeps only ckpt_*/ at iter >= this

ELICITATION_BUDGET = 56  # train-map total label budget (Stages 2/3/4)
ELICITATION_CAP    = 2   # max pairs per rollout
VAL_MAP_BUDGET     = 14  # val-map labels (random sampling, eval-only)

# Stage 4 (iterative)
N_ROUNDS     = 4   # rounds × budget = total budget; 4×14 = 56
ROUND_BUDGET = 14  # per-round labels
ROUND_EPOCHS = 8   # epochs per round (4 × 8 = 32, comparable to Stages 2/3's 30)

DEVICE = "cuda"

import os
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(ITERATIVE_DIR, exist_ok=True)
for p in ("REPO_DIR", "OUT_DIR", "BRANCH"):
    print(f"{p:14s} {globals()[p]}")

REPO_DIR       /content/MAPF-GPT-DDG
OUT_DIR        /content/drive/MyDrive/mapf_congestion/out/segment_classifier
BRANCH         feature/preference_elicitation


In [3]:
%cd /content
!rm -rf "$REPO_DIR"
!git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"
%cd "$REPO_DIR"
!git log --oneline -5

/content
Cloning into '/content/MAPF-GPT-DDG'...
remote: Enumerating objects: 2044, done.
remote: Counting objects: 100% (1893/1893), done.
remote: Compressing objects: 100% (1600/1600), done.
remote: Total 2044 (delta 335), reused 1800 (delta 286), pack-reused 151 (from 2)
Receiving objects: 100% (2044/2044), 79.81 MiB | 19.32 MiB/s, done.
Resolving deltas: 100% (364/364), done.
/content/MAPF-GPT-DDG
b2e8142 (HEAD -> feature/preference_elicitation, origin/feature/preference_elicitation) stage 3 and 4 training
ba00c92 add svg and thank you slides
3048f93 add presentation plan and intial slide deck
a32d7be Add AL to report
78d75ef 4 stages


In [4]:
# Colab already ships numpy, torch, matplotlib. Pin numpy<2 to match the rest of the project.
!python -m pip install -q --no-warn-conflicts "numpy<2" "matplotlib"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 127.8 MB/s eta 0:00:00


In [5]:
import torch
print('torch', torch.__version__)
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

torch 2.10.0+cu128
cuda True NVIDIA A100-SXM4-40GB


In [10]:
%%bash
# Sanity-check dataset wiring before launching the long training runs.
cd /content/MAPF-GPT-DDG
python - <<'PY'
from pathlib import Path
from train_segment_classifier import load_annotations, SegmentPairDataset
from held_out_seed_set import TRAIN_MAP_SEEDS, VAL_MAP_SEEDS

# Legacy annotations (Stage 1 post-hoc diagnostic only — never used as overrides).
ann = load_annotations('annotations_legacy_free_curation.json')
print(f'legacy annotations: {len(ann)} unique rollouts')
n_match = sum(1 for k in ann if Path(k).exists())
print(f'  on disk: {n_match}/{len(ann)}')

paths = list(Path('dataset/held_out').rglob('*.npz'))
print(f'episode files under dataset/held_out: {len(paths)}')

for name in ('train_seeds_manifest.txt', 'val_seeds_manifest.txt'):
    p = Path(name)
    if p.exists():
        n = sum(1 for line in p.read_text().splitlines() if line.strip() and not line.startswith('#'))
        print(f'{name}: {n} paths')
    else:
        print(f'{name}: <not built yet — run the manifest-build cells>')
PY


legacy annotations: 52 unique rollouts
  on disk: 52/52
episode files under dataset/held_out: 1669
train_seeds_manifest.txt: 1340 paths
val_seeds_manifest.txt: 329 paths


## Build elicitation manifests (train + val)

Stages 2/3/4 elicit human labels from the **train-map** pool only (so every label fires as a training override). The **val-map** pool is held out for human-aligned generalization eval — random-sampled once and passed to every stage as `--val-annotations`. Cell below builds both manifests in one pass.

In [ ]:
%%bash
cd /content/MAPF-GPT-DDG
python - <<'PY'
import numpy as np
from pathlib import Path
from held_out_seed_set import TRAIN_MAP_SEEDS, VAL_MAP_SEEDS

train_lines = ['# Train-map manifest (TRAIN_MAP_SEEDS=128-143). Source for Stages 2/3/4 elicitation.']
val_lines   = ['# Val-map manifest (VAL_MAP_SEEDS=144-147). Source for held-out human val (random sampling, eval-only).']
n_train = n_val = n_other = 0
for npz in sorted(Path('dataset/held_out').rglob('*.npz')):
    seed = int(np.load(str(npz), allow_pickle=True)['map_seed'])
    if seed in TRAIN_MAP_SEEDS:
        train_lines.append(str(npz)); n_train += 1
    elif seed in VAL_MAP_SEEDS:
        val_lines.append(str(npz)); n_val += 1
    else:
        n_other += 1
Path('train_seeds_manifest.txt').write_text('\n'.join(train_lines) + '\n')
Path('val_seeds_manifest.txt').write_text('\n'.join(val_lines) + '\n')
print(f'train_seeds_manifest.txt: {n_train} rollouts')
print(f'val_seeds_manifest.txt:   {n_val} rollouts')
print(f'(skipped {n_other} on other map seeds)')
PY


## Elicit val-map human labels (one-time, eval-only)

Before launching any stage, run a one-shot **random-sampling** elicitation against the val-map manifest. These labels are eval-only: every stage's training run will pass them via `--val-annotations`, never as overrides. Random sampling is the right protocol for val — uniformly distributed pairs give an unbiased estimate of human-aligned generalization, unlike acquisition functions which are biased toward "informative-to-current-model" pairs.

**Run locally:**
```
python query.py --manifest val_seeds_manifest.txt \
    --output annotation_val_map.json \
    --random --seed 7 \
    --budget 14 --per-episode-cap 2 \
    -r
```
Upload `annotation_val_map.json` to `VAL_MAP_ANNOTATIONS` on Drive. Done once — every stage below references it.

## Stage 1 — auto-only baseline

Train on auto-labels (`segment_diffs`) alone. No `--annotations` is passed — the model never sees human verdicts during training. `--val-annotations` provides the held-out val-map human signal for monitoring + best-ckpt selection. This is the auto-only floor we're trying to beat.

`baseline.pt` is saved as the best ckpt by `human_val` (val-map pairwise) when val-map labels are present; otherwise by auto `argmax_pm1`. Either way, this checkpoint feeds Stages 2/3/4 via `--init-from`.

Training time on Colab GPU: ~30–60 min.

In [11]:
%cd $REPO_DIR
!python train_segment_classifier.py \
    --data {DATA_DIR} \
    --output {BASELINE_CKPT} \
    --val-annotations {VAL_MAP_ANNOTATIONS} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --context_segments {CONTEXT_SEGMENTS} \
    --base_ch {BASE_CH} \
    --min_checkpoint {MIN_CHECKPOINT} \
    --device {DEVICE} 2>&1 | tee {OUT_DIR}/baseline.log

/content/MAPF-GPT-DDG
Found 1669 episode files under /content/MAPF-GPT-DDG/dataset/held_out
Val-map annotations: 12 rollouts (0 not on disk; eval-only)
Train pairs: 17728  |  Val pairs: 3652
Epoch   1 | lr 2.99e-04 | loss 0.5848 | top-1±1 0.538 | top-3 0.587 | hp tr/val 0.000/0.333 | h-arg t1 tr/val 0.000/0.000 | h-arg ±1 tr/val 0.000/0.167
  → saved baseline.argmax_pm1.pt (best argmax_pm1: 0.538)
  → saved baseline.pt (best human_val: 0.333)
  → saved baseline.human_argmax.pt (best human_argmax±1: 0.167)
Epoch   2 | lr 2.97e-04 | loss 0.5388 | top-1±1 0.556 | top-3 0.568 | hp tr/val 0.000/0.333 | h-arg t1 tr/val 0.000/0.000 | h-arg ±1 tr/val 0.000/0.000
  → saved baseline.argmax_pm1.pt (best argmax_pm1: 0.556)
Epoch   3 | lr 2.94e-04 | loss 0.5030 | top-1±1 0.565 | top-3 0.565 | hp tr/val 0.000/0.333 | h-arg t1 tr/val 0.000/0.000 | h-arg ±1 tr/val 0.000/0.000
  → saved baseline.argmax_pm1.pt (best argmax_pm1: 0.565)
Epoch   4 | lr 2.89e-04 | loss 0.4881 | top-1±1 0.565 | top-3 0.574 |

## Stage 2 — random-sampling baseline

Same one-pair-at-a-time labelling protocol as Stages 3/4, but acquisition is **uniform random sampling** across the train-map elicitation pool. This is the canonical baseline against which AL's value gets measured: if Stages 3/4 don't beat random, the acquisition function isn't doing real work.

**Run locally first** (interactive matplotlib + keyboard input):
```
python query.py --manifest train_seeds_manifest.txt \
    --output annotation_random.json \
    --random --seed 42 \
    --budget 56 \
    --per-episode-cap 2 \
    -r
```
Upload `annotation_random.json` to `RANDOM_ANNOTATIONS` on Drive, then run the cell below.

All 56 labels go into training overrides (no within-train holdout). The held-out human val signal — used for save-best ckpt selection — comes from `--val-annotations` (val-map labels collected once up-front).

In [22]:
!git pull

remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 71.57 KiB | 7.95 MiB/s, done.
From https://github.com/isa-deluis42/MAPF-GPT-DDG
   8ba3bd5..6de0a87  feature/preference_elicitation -> origin/feature/preference_elicitation
Updating 8ba3bd5..6de0a87
Fast-forward
 annotation_iterative.json         | 1036 +++++++++++++++++++++++++++++++++++++
 out/segment_classifier/round_3.pt |  Bin 0 -> 78585 bytes
 2 files changed, 1036 insertions(+)
 create mode 100644 out/segment_classifier/round_3.pt


In [13]:
%cd $REPO_DIR
!python train_segment_classifier.py \
    --data {DATA_DIR} \
    --output {RANDOM_CKPT} \
    --annotations {RANDOM_ANNOTATIONS} \
    --val-annotations {VAL_MAP_ANNOTATIONS} \
    --init-from {BASELINE_CKPT} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --context_segments {CONTEXT_SEGMENTS} \
    --base_ch {BASE_CH} \
    --min_checkpoint {MIN_CHECKPOINT} \
    --device {DEVICE} 2>&1 | tee {OUT_DIR}/random.log

/content/MAPF-GPT-DDG
Found 1669 episode files under /content/MAPF-GPT-DDG/dataset/held_out
Train-map annotations: 47 rollouts (0 not on disk; all used as training overrides)
Val-map annotations: 12 rollouts (0 not on disk; eval-only)
Train pairs: 16736  |  Val pairs: 3652
  ↳ 47 train episodes had auto pairs replaced by 47 human pair(s) total
Initialized weights from /content/drive/MyDrive/mapf_congestion/out/segment_classifier/baseline.pt (base_ch=8, ctx=1, save_by=human_val)
Epoch   1 | lr 2.99e-04 | loss 0.4362 | top-1±1 0.550 | top-3 0.550 | hp tr/val 0.489/0.417 | h-arg t1 tr/val 0.106/0.000 | h-arg ±1 tr/val 0.255/0.000
  → saved random_baseline.argmax_pm1.pt (best argmax_pm1: 0.550)
  → saved random_baseline.pt (best human_val: 0.417)
Epoch   2 | lr 2.97e-04 | loss 0.4309 | top-1±1 0.523 | top-3 0.562 | hp tr/val 0.511/0.333 | h-arg t1 tr/val 0.064/0.000 | h-arg ±1 tr/val 0.234/0.000
Epoch   3 | lr 2.94e-04 | loss 0.4232 | top-1±1 0.532 | top-3 0.553 | hp tr/val 0.511/0.333 | h

## Stage 3 — warm-start preference elicitation (pool-based AL)

Same pool, same protocol as Stage 2 — but the Stage 1 baseline scores every segment, and `query.py` ranks `(rollout, segA, segB)` candidates by `H(σ(s_A − s_B)) · ‖φ_A − φ_B‖` against globally-normalized features. The greedy selector picks the top-`--budget` candidates respecting `--per-episode-cap`.

The pool is **1340 rollouts** on TRAIN_MAP_SEEDS (segments range 2–16 across MAPF-GPT checkpoints, ~95k candidate pairs total). The acquisition function does the filtering: pre-filtering with a handcrafted predicate would smuggle in human priors that bypass the AL mechanism.

**Run locally first** (interactive matplotlib + keyboard input):
```
python query.py --manifest train_seeds_manifest.txt \
    --output annotation_elicitation_warmstart.json \
    --model-path out/segment_classifier/baseline.pt \
    --budget 56 \
    --per-episode-cap 2 \
    -r
```
Upload to `WARMSTART_ANNOTATIONS` on Drive.

In [15]:
%cd $REPO_DIR
!python train_segment_classifier.py \
    --data {DATA_DIR} \
    --output {WARMSTART_CKPT} \
    --annotations {WARMSTART_ANNOTATIONS} \
    --val-annotations {VAL_MAP_ANNOTATIONS} \
    --init-from {BASELINE_CKPT} \
    --epochs {EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --context_segments {CONTEXT_SEGMENTS} \
    --base_ch {BASE_CH} \
    --min_checkpoint {MIN_CHECKPOINT} \
    --device {DEVICE} 2>&1 | tee {OUT_DIR}/warmstart.log

/content/MAPF-GPT-DDG
Found 1669 episode files under /content/MAPF-GPT-DDG/dataset/held_out
Train-map annotations: 35 rollouts (0 not on disk; all used as training overrides)
Val-map annotations: 12 rollouts (0 not on disk; eval-only)
Train pairs: 17313  |  Val pairs: 3652
  ↳ 35 train episodes had auto pairs replaced by 56 human pair(s) total
Initialized weights from /content/drive/MyDrive/mapf_congestion/out/segment_classifier/baseline.pt (base_ch=8, ctx=1, save_by=human_val)
Epoch   1 | lr 2.99e-04 | loss 0.4319 | top-1±1 0.544 | top-3 0.562 | human_val tr/val 0.268/0.333
  → saved warmstart_elicitation.argmax_pm1.pt (best argmax_pm1: 0.544)
  → saved warmstart_elicitation.pt (best human_val: 0.333)
Epoch   2 | lr 2.97e-04 | loss 0.4273 | top-1±1 0.562 | top-3 0.581 | human_val tr/val 0.232/0.333
  → saved warmstart_elicitation.argmax_pm1.pt (best argmax_pm1: 0.562)
Epoch   3 | lr 2.94e-04 | loss 0.4259 | top-1±1 0.541 | top-3 0.541 | human_val tr/val 0.304/0.500
  → saved warmstart

## Stage 4 — iterative active learning (rounds-based)

Replaces one-shot AL with **R=4 rounds × 14 labels = 56 total** (matches Stages 2/3 budget). Each round: query 14 new pairs using the *current* model (`baseline.pt` for round 1, `round_{r-1}.pt` thereafter), then continue training that model for 8 epochs on the cumulative annotation set. Continue-from-prev-checkpoint (rather than restart-from-baseline) accumulates the fine-tuning so each round's compute actually moves the final model forward, not just the next round's query selection.

**Workflow** — labelling happens locally (interactive matplotlib + stdin), training happens here in Colab. Per round:

1. **Locally:** download the prev-round ckpt from Drive (`baseline.pt` for round 1, then `round_{r-1}.pt`), then run:
   ```
   python query.py --manifest train_seeds_manifest.txt \
       --model-path <prev_ckpt>.pt \
       --budget 14 --per-episode-cap 2 \
       --output annotation_iterative.json
   ```
   `query.py` is resumable: it skips already-labelled pairs and enforces `--per-episode-cap` cumulatively across rounds, so no rollout ever sees >2 queries even spanning all four rounds.
2. **Upload** the updated `annotation_iterative.json` to `ITERATIVE_ANNOTATIONS` on Drive.
3. **Run that round's training cell below.**
4. Repeat with the new round's ckpt.

After round 4, `ITERATIVE_DIR/round_4.pt` is the final Stage 4 model and `annotation_iterative.json` contains all 56 labels (14 per round, no overlap).

### Round 1 — first 14 labels driven by Stage 1 baseline

**Local first:** download `baseline.pt`, then:
```
python query.py --manifest train_seeds_manifest.txt \
    --model-path baseline.pt \
    --budget 14 --per-episode-cap 2 \
    --output annotation_iterative.json
```
Upload `annotation_iterative.json` to Drive, then run the cell below.

In [17]:
%cd $REPO_DIR
!python train_segment_classifier.py \
    --data {DATA_DIR} \
    --output {ITERATIVE_DIR}/round_1.pt \
    --annotations {ITERATIVE_ANNOTATIONS} \
    --val-annotations {VAL_MAP_ANNOTATIONS} \
    --init-from {BASELINE_CKPT} \
    --epochs {ROUND_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --context_segments {CONTEXT_SEGMENTS} \
    --base_ch {BASE_CH} \
    --min_checkpoint {MIN_CHECKPOINT} \
    --device {DEVICE} 2>&1 | tee {OUT_DIR}/iterative_round_1.log

/content/MAPF-GPT-DDG
Found 1669 episode files under /content/MAPF-GPT-DDG/dataset/held_out
Train-map annotations: 8 rollouts (0 not on disk; all used as training overrides)
Val-map annotations: 12 rollouts (0 not on disk; eval-only)
Train pairs: 17583  |  Val pairs: 3652
  ↳ 8 train episodes had auto pairs replaced by 14 human pair(s) total
Initialized weights from /content/drive/MyDrive/mapf_congestion/out/segment_classifier/baseline.pt (base_ch=8, ctx=1, save_by=human_val)
Epoch   1 | lr 2.89e-04 | loss 0.4337 | top-1±1 0.544 | top-3 0.562 | human_val tr/val 0.286/0.250
  → saved round_1.argmax_pm1.pt (best argmax_pm1: 0.544)
  → saved round_1.pt (best human_val: 0.250)
Epoch   2 | lr 2.56e-04 | loss 0.4274 | top-1±1 0.559 | top-3 0.565 | human_val tr/val 0.143/0.333
  → saved round_1.argmax_pm1.pt (best argmax_pm1: 0.559)
  → saved round_1.pt (best human_val: 0.333)
Epoch   3 | lr 2.07e-04 | loss 0.4246 | top-1±1 0.556 | top-3 0.571 | human_val tr/val 0.286/0.500
  → saved round_1.

### Round 2 — next 14 labels driven by `round_1.pt`

**Local first:** download `ITERATIVE_DIR/round_1.pt`, then:
```
python query.py --manifest train_seeds_manifest.txt \
    --model-path round_1.pt \
    --budget 14 --per-episode-cap 2 \
    --output annotation_iterative.json
```
The same `annotation_iterative.json` accumulates — query.py appends 14 new pairs without re-asking the round-1 ones. Upload, then run the cell below.

In [19]:
%cd $REPO_DIR
!python train_segment_classifier.py \
    --data {DATA_DIR} \
    --output {ITERATIVE_DIR}/round_2.pt \
    --annotations {ITERATIVE_ANNOTATIONS} \
    --val-annotations {VAL_MAP_ANNOTATIONS} \
    --init-from {ITERATIVE_DIR}/round_1.pt \
    --epochs {ROUND_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --context_segments {CONTEXT_SEGMENTS} \
    --base_ch {BASE_CH} \
    --min_checkpoint {MIN_CHECKPOINT} \
    --device {DEVICE} 2>&1 | tee {OUT_DIR}/iterative_round_2.log

/content/MAPF-GPT-DDG
Found 1669 episode files under /content/MAPF-GPT-DDG/dataset/held_out
Train-map annotations: 17 rollouts (0 not on disk; all used as training overrides)
Val-map annotations: 12 rollouts (0 not on disk; eval-only)
Train pairs: 17484  |  Val pairs: 3652
  ↳ 17 train episodes had auto pairs replaced by 28 human pair(s) total
Initialized weights from /content/drive/MyDrive/mapf_congestion/out/segment_classifier/iterative/round_1.pt (base_ch=8, ctx=1, save_by=human_val)
Epoch   1 | lr 2.89e-04 | loss 0.4190 | top-1±1 0.550 | top-3 0.562 | human_val tr/val 0.321/0.333
  → saved round_2.argmax_pm1.pt (best argmax_pm1: 0.550)
  → saved round_2.pt (best human_val: 0.333)
Epoch   2 | lr 2.56e-04 | loss 0.4192 | top-1±1 0.541 | top-3 0.574 | human_val tr/val 0.250/0.500
  → saved round_2.pt (best human_val: 0.500)
Epoch   3 | lr 2.07e-04 | loss 0.4117 | top-1±1 0.574 | top-3 0.590 | human_val tr/val 0.250/0.333
  → saved round_2.argmax_pm1.pt (best argmax_pm1: 0.574)
Epoch  

### Round 3 — next 14 labels driven by `round_2.pt`

Same pattern. Local: query with `round_2.pt`, append to `annotation_iterative.json`, upload, then train below.

In [21]:
%cd $REPO_DIR
!python train_segment_classifier.py \
    --data {DATA_DIR} \
    --output {ITERATIVE_DIR}/round_3.pt \
    --annotations {ITERATIVE_ANNOTATIONS} \
    --val-annotations {VAL_MAP_ANNOTATIONS} \
    --init-from {ITERATIVE_DIR}/round_2.pt \
    --epochs {ROUND_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --context_segments {CONTEXT_SEGMENTS} \
    --base_ch {BASE_CH} \
    --min_checkpoint {MIN_CHECKPOINT} \
    --device {DEVICE} 2>&1 | tee {OUT_DIR}/iterative_round_3.log

/content/MAPF-GPT-DDG
Found 1669 episode files under /content/MAPF-GPT-DDG/dataset/held_out
Train-map annotations: 25 rollouts (0 not on disk; all used as training overrides)
Val-map annotations: 12 rollouts (0 not on disk; eval-only)
Train pairs: 17424  |  Val pairs: 3652
  ↳ 25 train episodes had auto pairs replaced by 42 human pair(s) total
Initialized weights from /content/drive/MyDrive/mapf_congestion/out/segment_classifier/iterative/round_2.pt (base_ch=8, ctx=1, save_by=human_val)
Epoch   1 | lr 2.89e-04 | loss 0.4151 | top-1±1 0.565 | top-3 0.596 | human_val tr/val 0.786/0.417
  → saved round_3.argmax_pm1.pt (best argmax_pm1: 0.565)
  → saved round_3.pt (best human_val: 0.417)
Epoch   2 | lr 2.56e-04 | loss 0.4092 | top-1±1 0.553 | top-3 0.581 | human_val tr/val 0.714/0.500
  → saved round_3.pt (best human_val: 0.500)
Epoch   3 | lr 2.07e-04 | loss 0.4053 | top-1±1 0.559 | top-3 0.578 | human_val tr/val 0.405/0.417
Epoch   4 | lr 1.50e-04 | loss 0.4012 | top-1±1 0.550 | top-3 0.

### Round 4 — final 14 labels driven by `round_3.pt`

Last round. Local: query with `round_3.pt`, append, upload, then train below. The resulting `round_4.pt` is the final Stage 4 model.

In [23]:
%cd $REPO_DIR
!python train_segment_classifier.py \
    --data {DATA_DIR} \
    --output {ITERATIVE_DIR}/round_4.pt \
    --annotations {ITERATIVE_ANNOTATIONS} \
    --val-annotations {VAL_MAP_ANNOTATIONS} \
    --init-from {ITERATIVE_DIR}/round_3.pt \
    --epochs {ROUND_EPOCHS} \
    --batch_size {BATCH_SIZE} \
    --lr {LR} \
    --context_segments {CONTEXT_SEGMENTS} \
    --base_ch {BASE_CH} \
    --min_checkpoint {MIN_CHECKPOINT} \
    --device {DEVICE} 2>&1 | tee {OUT_DIR}/iterative_round_4.log

/content/MAPF-GPT-DDG
Found 1669 episode files under /content/MAPF-GPT-DDG/dataset/held_out
Train-map annotations: 34 rollouts (0 not on disk; all used as training overrides)
Val-map annotations: 12 rollouts (0 not on disk; eval-only)
Train pairs: 17281  |  Val pairs: 3652
  ↳ 34 train episodes had auto pairs replaced by 56 human pair(s) total
Initialized weights from /content/drive/MyDrive/mapf_congestion/out/segment_classifier/iterative/round_3.pt (base_ch=8, ctx=1, save_by=human_val)
Epoch   1 | lr 2.89e-04 | loss 0.4044 | top-1±1 0.535 | top-3 0.578 | human_val tr/val 0.429/0.417
  → saved round_4.argmax_pm1.pt (best argmax_pm1: 0.535)
  → saved round_4.pt (best human_val: 0.417)
Epoch   2 | lr 2.56e-04 | loss 0.4007 | top-1±1 0.535 | top-3 0.559 | human_val tr/val 0.607/0.500
  → saved round_4.pt (best human_val: 0.500)
Epoch   3 | lr 2.07e-04 | loss 0.3975 | top-1±1 0.541 | top-3 0.559 | human_val tr/val 0.804/0.500
  → saved round_4.argmax_pm1.pt (best argmax_pm1: 0.541)
Epoch  

## Post-hoc: legacy human-pair diagnostic on Stage 1 baseline

Auxiliary diagnostic — loads `baseline.pt` from Drive and reports its agreement with the legacy 52-label hand-curated set (free-curation protocol, all on TRAIN_MAP_SEEDS). Useful as a sanity check that the auto baseline isn't catastrophically misaligned with humans, but **not** part of the four-stage comparison (different protocol, different rollouts than the val-map signal). Re-runnable with any saved Stage 1 ckpt.


In [ ]:
%%bash
cd /content/MAPF-GPT-DDG
python - <<'PY'
import torch
from pathlib import Path
from train_segment_classifier import (
    Segment3DCNN, evaluate_metrics, evaluate_human_pairs, load_annotations,
)
from held_out_seed_set import VAL_MAP_SEEDS

BASELINE_PT = '/content/drive/MyDrive/mapf_congestion/out/segment_classifier/baseline.pt'
LEGACY_ANNOTATIONS = 'annotations_legacy_free_curation.json'

ckpt = torch.load(BASELINE_PT, map_location='cuda', weights_only=False)
ctx  = ckpt.get('context_segments', 1)
model = Segment3DCNN(base_ch=ckpt['base_ch']).cuda()
model.load_state_dict(ckpt['state_dict'])

paths = list(Path('dataset/held_out').rglob('*.npz'))
print(f'episode files: {len(paths)}; ckpt context_segments={ctx}')

pm1, top3 = evaluate_metrics(model, paths, VAL_MAP_SEEDS, 'cuda', context_segments=ctx)
print(f'Stage 1 baseline.pt — auto val: top-1±1={pm1:.3f}  top-3={top3:.3f}')

ann = load_annotations(LEGACY_ANNOTATIONS)
acc, n = evaluate_human_pairs(model, ann, 'cuda', context_segments=ctx)
print(f'  legacy human-pair acc (52 train-map labels, free-curation protocol): {acc:.3f} ({n})')
PY


## Compare

In [ ]:
# Pull per-epoch metrics out of all four training logs and compare.
import re, pathlib

# New per-epoch line format:
# Epoch X | lr Y | loss Z | top-1±1 P | top-3 T | hp tr/val A/B | h-arg t1 tr/val X/Y | h-arg ±1 tr/val P/Q
LINE = re.compile(
    r"Epoch\s+(\d+)\s+\|(?:\s+lr\s+[\d.eE+-]+\s+\|)?\s+loss\s+([\d.]+)"
    r"\s+\|\s+top-1.1\s+([\d.]+)\s+\|\s+top-3\s+([\d.]+)"
    r"(?:.*hp\s+tr/val\s+([\d.]+)/([\d.]+))?"
    r"(?:.*h-arg\s+t1\s+tr/val\s+([\d.]+)/([\d.]+))?"
    r"(?:.*h-arg\s+.1\s+tr/val\s+([\d.]+)/([\d.]+))?"
)

def parse_log(path):
    rows = []
    if not pathlib.Path(path).exists():
        print(f"[skip] log not found: {path}")
        return rows
    for line in pathlib.Path(path).read_text().splitlines():
        m = LINE.search(line)
        if not m: continue
        g = m.groups()
        row = {"epoch": int(g[0]), "loss": float(g[1]),
               "pm1": float(g[2]), "top3": float(g[3])}
        if g[4] is not None:
            row.update({"hp_tr": float(g[4]), "hp_val": float(g[5])})
        if g[6] is not None:
            row.update({"t1_tr": float(g[6]), "t1_val": float(g[7])})
        if g[8] is not None:
            row.update({"pm1h_tr": float(g[8]), "pm1h_val": float(g[9])})
        rows.append(row)
    return rows

stages = [
    ("Stage 1 auto",       f"{OUT_DIR}/baseline.log",            "#666666"),
    ("Stage 2 random",     f"{OUT_DIR}/random.log",              "#1f77b4"),
    ("Stage 3 warmstart",  f"{OUT_DIR}/warmstart.log",           "#2ca02c"),
    ("Stage 4 iterative",  f"{OUT_DIR}/iterative_round_4.log",   "#d62728"),
]
parsed = [(label, parse_log(path), color) for label, path, color in stages]

def best(rows, key):
    return max((r.get(key, 0.0) for r in rows), default=0.0)

print(f"=== Best metrics per stage ===")
hdr = f"{'metric':28s}  " + "  ".join(f"{label:>20s}" for label, _, _ in parsed)
print(hdr); print("-" * len(hdr))
for label, key in [
    ("val top-1±1 (auto, val-map)",      "pm1"),
    ("val top-3 (auto, val-map)",         "top3"),
    ("hp pairwise val (val-map)",         "hp_val"),
    ("hp pairwise tr (sanity)",           "hp_tr"),
    ("h-arg top1 val (val-map)",          "t1_val"),
    ("h-arg ±1 val (val-map)",            "pm1h_val"),
]:
    cells = "  ".join(f"{best(rows,key):>20.3f}" for _, rows, _ in parsed)
    print(f"{label:28s}  {cells}")

print()
print("=== Saved checkpoints ===")
import os
expected = [
    f"{OUT_DIR}/baseline.pt",
    f"{OUT_DIR}/random_baseline.pt",
    f"{OUT_DIR}/warmstart_elicitation.pt",
] + [f"{ITERATIVE_DIR}/round_{r}.pt" for r in range(1, N_ROUNDS + 1)]
for p in expected:
    print(f"  {os.path.basename(p):30s}  {'exists' if os.path.exists(p) else '<missing>'}")

# Plot 1: best-of-stage comparison
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for label, rows, color in parsed:
    if not rows: continue
    ep = [r["epoch"] for r in rows]
    if rows and "hp_val" in rows[0]:
        axes[0].plot(ep, [r["hp_val"] for r in rows], label=label, color=color)
    if rows and "t1_val" in rows[0]:
        axes[1].plot(ep, [r["t1_val"] for r in rows], label=label, color=color)
    axes[2].plot(ep, [r["pm1"] for r in rows], label=label, color=color)
axes[0].axhline(0.5,   color="red", linestyle=":", alpha=0.4, label="chance")
axes[0].set_ylabel("HP Pairwise val (val-map)"); axes[0].set_xlabel("Epoch"); axes[0].grid(True); axes[0].legend(); axes[0].set_ylim(0,1)
axes[1].set_ylabel("HP Argmax top-1 val (val-map)"); axes[1].set_xlabel("Epoch"); axes[1].grid(True); axes[1].legend(); axes[1].set_ylim(0,1)
axes[2].set_ylabel("Auto val top-1±1 (val-map)"); axes[2].set_xlabel("Epoch"); axes[2].grid(True); axes[2].legend(); axes[2].set_ylim(0,1)
fig.suptitle("Stage comparison (val signals on val-map rollouts)")
fig.tight_layout()
plt.savefig(f"{OUT_DIR}/comparison.png", dpi=150)
plt.show()

# Plot 2: per-round iterative trajectory (Stage 4) — does each round buy improvement?
round_logs = [(r, parse_log(f"{OUT_DIR}/iterative_round_{r}.log")) for r in range(1, N_ROUNDS + 1)]
if any(rows for _, rows in round_logs):
    fig2, axes2 = plt.subplots(1, 2, figsize=(12, 4))
    cumulative_epoch = 0
    for r, rows in round_logs:
        if not rows: continue
        offset_eps = [cumulative_epoch + e["epoch"] for e in rows]
        axes2[0].plot(offset_eps, [e.get("hp_val", float("nan")) for e in rows], label=f"round {r}")
        axes2[1].plot(offset_eps, [e.get("pm1", float("nan")) for e in rows], label=f"round {r}")
        cumulative_epoch += max((e["epoch"] for e in rows), default=0)
    axes2[0].set_ylabel("HP Pairwise val (val-map)"); axes2[0].set_xlabel("Cumulative epoch"); axes2[0].grid(True); axes2[0].legend(); axes2[0].set_ylim(0,1)
    axes2[1].set_ylabel("Auto val top-1±1 (val-map)"); axes2[1].set_xlabel("Cumulative epoch"); axes2[1].grid(True); axes2[1].legend(); axes2[1].set_ylim(0,1)
    fig2.suptitle("Stage 4 iterative AL — per-round trajectory")
    fig2.tight_layout()
    plt.savefig(f"{OUT_DIR}/iterative_trajectory.png", dpi=150)
    plt.show()
